# Notebook 1 — SQLite Schema + Seed Data

Pure SQL. No AI. Goal: verify all queries return correct results before adding any intelligence.

**Exit criteria:** All test queries return correct expected results.

## 1. Setup — Create Database

In [1]:
import sqlite3, os
from datetime import datetime

# Use a file-based DB so other notebooks can reuse it.
# Delete any existing file first so re-running this notebook is idempotent —
# otherwise the unconditional INSERTs below double the seed rows on every Run All
# (Feb total goes 3506 -> 7012 -> 10518, Maddy 7000 -> 14000, etc.).
DB_PATH = "second_brain.db"
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)
    print(f"Removed stale {DB_PATH}")
conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row  # access columns by name
cur = conn.cursor()
print(f"Database created at: {DB_PATH}")

Removed stale second_brain.db
Database created at: second_brain.db


## 2. Create Tables

In [ ]:
cur.executescript('''
-- Persons: single source of truth for known people. Used by weights and ledger.
-- Managed exclusively via chat commands ADD_PERSON / REMOVE_PERSON / MODIFY_PERSON
-- or via the People screen in the Flask app.
CREATE TABLE IF NOT EXISTS persons (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL UNIQUE,
    created_at TEXT DEFAULT (datetime('now'))
);

-- Activity log: persistent record of every input + response.
-- Powers the home feed (last 10) and the /activity page (full history).
CREATE TABLE IF NOT EXISTS activity_log (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    input_text TEXT NOT NULL,
    response_text TEXT NOT NULL,
    kind TEXT,        -- 'query' | 'write' | 'person_command' | 'unknown'
    created_at TEXT DEFAULT (datetime('now', 'localtime'))
);

-- Expenses (no category column — description used verbatim with LIKE for ad-hoc filtering)
CREATE TABLE IF NOT EXISTS expenses (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    amount REAL NOT NULL,
    description TEXT NOT NULL,
    date TEXT,
    month TEXT,
    raw_note TEXT,
    created_at TEXT DEFAULT (datetime('now'))
);

-- Ledger (append-only, never UPDATE)
CREATE TABLE IF NOT EXISTS ledger (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    person TEXT NOT NULL,
    amount REAL NOT NULL,
    direction TEXT NOT NULL,  -- gave | received
    note TEXT,
    date TEXT,
    created_at TEXT DEFAULT (datetime('now'))
);

-- Ledger balance view (positive = they owe you, negative = you owe them)
CREATE VIEW IF NOT EXISTS ledger_balance AS
SELECT
    person,
    SUM(CASE WHEN direction = 'gave' THEN amount ELSE -amount END) AS balance
FROM ledger
GROUP BY person;

-- Weights
CREATE TABLE IF NOT EXISTS weights (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    person TEXT NOT NULL,
    weight REAL NOT NULL,
    date TEXT NOT NULL,
    note TEXT,
    created_at TEXT DEFAULT (datetime('now'))
);

-- Todos
CREATE TABLE IF NOT EXISTS todos (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    content TEXT NOT NULL,
    date TEXT,
    status TEXT DEFAULT 'pending',
    created_at TEXT DEFAULT (datetime('now'))
);

-- Investment events
CREATE TABLE IF NOT EXISTS investment_events (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    symbol TEXT,
    event_type TEXT,
    content TEXT,
    amount REAL,
    date TEXT,
    source TEXT,
    created_at TEXT DEFAULT (datetime('now'))
);

-- Vector store for RAG
CREATE TABLE IF NOT EXISTS embeddings (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    domain TEXT NOT NULL,
    content TEXT NOT NULL,
    embedding BLOB NOT NULL,
    source TEXT,
    date TEXT,
    created_at TEXT DEFAULT (datetime('now'))
);
''')
conn.commit()
print("All tables created successfully.")

## 3. Seed Data

In [ ]:
# Persons whitelist (managed via chat commands later)
cur.executemany(
    "INSERT INTO persons (name) VALUES (?)",
    [('jeevi',), ('prani',), ('murugan',), ('maddy',), ('thenna',)]
)

# Ledger seed
cur.executemany(
    "INSERT INTO ledger (person, amount, direction, note) VALUES (?, ?, ?, ?)",
    [
        ('thenna', 20000, 'gave', 'initial balance'),
        ('maddy',   7000, 'gave', 'initial balance'),
    ]
)

# Jeevi weights
cur.executemany(
    "INSERT INTO weights (person, weight, date, note) VALUES (?, ?, ?, ?)",
    [
        ('jeevi', 60.1, '2026-04-24', None),
        ('jeevi', 59.1, '2026-02-27', None),
        ('jeevi', 58.7, '2026-02-18', None),
        ('jeevi', 57.2, '2026-01-26', None),
        ('jeevi', 57.8, '2026-01-20', None),
        ('jeevi', 57.9, '2025-12-30', None),
        ('jeevi', 58.7, '2025-12-21', None),
        ('jeevi', 58.1, '2025-12-08', 'night'),
        ('jeevi', 57.4, '2025-11-27', 'empty stomach'),
    ]
)

# Prani weights
cur.executemany(
    "INSERT INTO weights (person, weight, date, note) VALUES (?, ?, ?, ?)",
    [
        ('prani', 11.3, '2026-04-24', None),
        ('prani', 11.5, '2026-04-09', None),
        ('prani', 11.2, '2026-02-27', None),
        ('prani', 10.9, '2026-02-18', None),
        ('prani', 10.9, '2026-01-26', None),
        ('prani', 10.7, '2026-01-20', None),
        ('prani', 10.4, '2025-12-30', None),
        ('prani', 10.3, '2025-12-21', None),
        ('prani', 10.1, '2025-12-08', 'night'),
    ]
)

# Murugan weights
cur.executemany(
    "INSERT INTO weights (person, weight, date, note) VALUES (?, ?, ?, ?)",
    [
        ('murugan', 65.0, '2026-04-24', None),
        ('murugan', 64.7, '2026-04-09', None),
        ('murugan', 65.7, '2026-02-27', None),
        ('murugan', 65.4, '2026-02-19', None),
        ('murugan', 65.6, '2026-01-26', None),
        ('murugan', 66.1, '2026-01-20', None),
        ('murugan', 65.5, '2025-12-30', None),
        ('murugan', 66.2, '2025-12-21', None),
        ('murugan', 65.8, '2025-12-08', 'night'),
        ('murugan', 65.5, '2025-11-27', 'empty stomach'),
        ('murugan', 64.6, '2025-11-22', None),
    ]
)

# Sample expenses (no category — just amount, description, date, month)
cur.executemany(
    "INSERT INTO expenses (amount, description, date, month) VALUES (?, ?, ?, ?)",
    [
        (30,   'tea + bonda',                 '2026-02-01', '2026-02'),
        (184,  'mushroom fried rice',          '2026-02-01', '2026-02'),
        (650,  'Tirupathi undiyal',            '2026-02-01', '2026-02'),
        (500,  'Tirupathi zoo ebike',          '2026-02-01', '2026-02'),
        (143,  'Medplus Pampers',              '2026-02-01', '2026-02'),
        (999,  'jeevi dress',                  '2026-02-01', '2026-02'),
        (1000, 'petrol',                       '2026-02-01', '2026-02'),
        (2000, 'petrol',                       '2026-03-01', '2026-03'),
        (839,  'Bombay Ananda bhavan sweet',   '2026-03-01', '2026-03'),
    ]
)

# Sample todos
cur.executemany(
    "INSERT INTO todos (content, status) VALUES (?, ?)",
    [
        ('pay electricity bill', 'pending'),
        ('call for kadai 2 - need to prepare data', 'pending'),
        ('check whether falling market and gold rise can be used to advantage', 'pending'),
        ('Reddit personal finance india follow', 'done'),
    ]
)

conn.commit()
print("Seed data inserted.")

## 4. Test Queries — Verify All Results

In [ ]:
def run_test(label, query, params=(), expected_note=""):
    cur.execute(query, params)
    rows = cur.fetchall()
    print(f"\n{'='*50}")
    print(f"TEST: {label}")
    if expected_note:
        print(f"Expected: {expected_note}")
    print("Result:")
    for row in rows:
        print(dict(row))
    return rows

# 1. Maddy balance
run_test("Maddy balance",
    "SELECT person, balance FROM ledger_balance WHERE person = 'maddy'",
    expected_note="balance = 7000 (they owe you)")

# 2. Thenna balance
run_test("Thenna balance",
    "SELECT person, balance FROM ledger_balance WHERE person = 'thenna'",
    expected_note="balance = 20000")

# 3. Who owes me
run_test("Who owes me money",
    "SELECT person, balance FROM ledger_balance WHERE balance > 0 ORDER BY balance DESC",
    expected_note="Both Maddy (7000) and Thenna (20000)")

# 4. Jeevi latest weight
run_test("Jeevi latest weight",
    "SELECT weight, date, note FROM weights WHERE person = 'jeevi' ORDER BY date DESC LIMIT 1",
    expected_note="60.1 on 2026-04-24")

# 5. Prani last 5 weights
run_test("Prani last 5 weights",
    "SELECT weight, date FROM weights WHERE person = 'prani' ORDER BY date DESC LIMIT 5",
    expected_note="5 most recent prani entries")

# 6. Murugan weight trend (all, ascending)
run_test("Murugan weight trend",
    "SELECT weight, date FROM weights WHERE person = 'murugan' ORDER BY date ASC",
    expected_note="All murugan entries oldest first")

# 7. Feb 2026 total spend
run_test("Feb 2026 total spend",
    "SELECT SUM(amount) as total FROM expenses WHERE month = '2026-02'",
    expected_note="Sum of all Feb expenses: 3506")

# 8. Petrol spend Feb 2026 — uses LIKE on description (no category column)
run_test("Petrol spend Feb 2026",
    "SELECT SUM(amount) as total FROM expenses WHERE description LIKE '%petrol%' AND month = '2026-02'",
    expected_note="1000 (petrol only in Feb)")

# 9. Pending todos
run_test("Pending todos",
    "SELECT content, status FROM todos WHERE status = 'pending'",
    expected_note="3 pending todos")

# 10. Simulate reduce 6k from Maddy then check balance
print("\n" + "="*50)
print("TEST: Reduce 6k from Maddy")
cur.execute("INSERT INTO ledger (person, amount, direction, note) VALUES ('maddy', 6000, 'received', 'test reduction')")
conn.commit()
cur.execute("SELECT person, balance FROM ledger_balance WHERE person = 'maddy'")
row = dict(cur.fetchone())
print(f"Result: {row}")
print(f"Expected: balance = 1000")
assert row['balance'] == 1000, f"FAIL: expected 1000, got {row['balance']}"
print("PASS ✓")

# Rollback the test insert
cur.execute("DELETE FROM ledger WHERE note = 'test reduction'")
conn.commit()
print("\nTest row removed. Maddy balance restored to 7000.")

## 5. Exit Criteria Check

In [5]:

print("Running exit criteria checks...")
errors = []

# Check ledger
cur.execute("SELECT balance FROM ledger_balance WHERE person = 'maddy'")
r = cur.fetchone()
if not r or r['balance'] != 7000:
    errors.append("Maddy balance incorrect")

cur.execute("SELECT balance FROM ledger_balance WHERE person = 'thenna'")
r = cur.fetchone()
if not r or r['balance'] != 20000:
    errors.append("Thenna balance incorrect")

# Check weights
cur.execute("SELECT weight FROM weights WHERE person = 'jeevi' ORDER BY date DESC LIMIT 1")
r = cur.fetchone()
if not r or r['weight'] != 60.1:
    errors.append("Jeevi latest weight incorrect")

cur.execute("SELECT COUNT(*) as cnt FROM weights WHERE person = 'prani'")
r = cur.fetchone()
if r['cnt'] != 9:
    errors.append(f"Prani weight count wrong: {r['cnt']}")

# Check expenses
cur.execute("SELECT SUM(amount) as total FROM expenses WHERE month = '2026-02'")
r = cur.fetchone()
if not r or r['total'] != 3506:
    errors.append(f"Feb total incorrect: {r['total']}")

# Check todos
cur.execute("SELECT COUNT(*) as cnt FROM todos WHERE status = 'pending'")
r = cur.fetchone()
if r['cnt'] != 3:
    errors.append(f"Pending todos count wrong: {r['cnt']}")

if errors:
    print("FAILURES:")
    for e in errors:
        print(f"  ✗ {e}")
else:
    print("All exit criteria PASSED ✓")
    print("Ready for Notebook 2.")

conn.close()


Running exit criteria checks...
All exit criteria PASSED ✓
Ready for Notebook 2.
